In [1]:
from ultralytics import YOLO
import cv2
import numpy as np
import pandas as pd
import os


In [3]:
model=YOLO("best.pt")


In [8]:
IMAGE_FOLDER = "My-First-Project-1/train/images"


In [11]:
feature_data=[]
print("Starting feature extraction using segmentation...")

Starting feature extraction using segmentation...


In [16]:
for img_name in os.listdir(IMAGE_FOLDER):

    img_path = os.path.join(IMAGE_FOLDER, img_name)

    results = model(img_path)
    r = results[0]

    if r.masks is None:
        continue

    image = cv2.imread(img_path)
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

    # Compute gradient ONCE per image
    gradient = cv2.Laplacian(gray, cv2.CV_64F)

    masks = r.masks.data.cpu().numpy()

    for mask in masks:

        # Resize mask to match original image size
        mask = cv2.resize(mask, (gray.shape[1], gray.shape[0]))
        mask = mask > 0.5   # Convert to boolean

        mango_pixels = gray[mask]

        if len(mango_pixels) == 0:
            continue

        mean_T = np.mean(mango_pixels)
        max_T = np.max(mango_pixels)
        std_T = np.std(mango_pixels)
        var_T = np.var(mango_pixels)

        threshold = mean_T + 1.5 * std_T
        hotspot_ratio = np.sum(mango_pixels > threshold) / len(mango_pixels)

        gradient_mean = np.mean(np.abs(gradient[mask]))

        feature_data.append([
            img_name,
            mean_T,
            max_T,
            std_T,
            var_T,
            hotspot_ratio,
            gradient_mean
        ])


image 1/1 c:\Users\aksha\OneDrive\Desktop\mango_class\My-First-Project-1\train\images\FLIR0011_jpg.rf.1df6fe45198b6fdbbf087cd3d2dca09b.jpg: 640x640 1 mango, 108.8ms
Speed: 4.2ms preprocess, 108.8ms inference, 4.0ms postprocess per image at shape (1, 3, 640, 640)

image 1/1 c:\Users\aksha\OneDrive\Desktop\mango_class\My-First-Project-1\train\images\FLIR0011_jpg.rf.1e3a3f4c0e135fd1816eabc0a683a58b.jpg: 640x640 1 mango, 152.6ms
Speed: 6.3ms preprocess, 152.6ms inference, 6.2ms postprocess per image at shape (1, 3, 640, 640)

image 1/1 c:\Users\aksha\OneDrive\Desktop\mango_class\My-First-Project-1\train\images\FLIR0011_jpg.rf.20f0b633e8c37f08608654a804acd9bd.jpg: 640x640 1 mango, 166.0ms
Speed: 6.8ms preprocess, 166.0ms inference, 4.6ms postprocess per image at shape (1, 3, 640, 640)

image 1/1 c:\Users\aksha\OneDrive\Desktop\mango_class\My-First-Project-1\train\images\FLIR0011_jpg.rf.255969173f316f8869269e225db745f3.jpg: 640x640 1 mango, 155.9ms
Speed: 6.8ms preprocess, 155.9ms inference

In [19]:
columns = [
    "image_name",
    "mean_T",
    "max_T",
    "std_T",
    "var_T",
    "hotspot_ratio",
    "gradient_mean"
]
df = pd.DataFrame(feature_data, columns=columns)

df.to_csv("segmentation_features.csv", index=False)

print("Done.")
print("Total mango samples:", len(df))

Done.
Total mango samples: 4260


In [20]:
import pandas as pd

df = pd.read_csv("segmentation_features.csv")

print(df.head())
print(df.describe())

                                          image_name      mean_T  max_T  \
0  FLIR0011_jpg.rf.1df6fe45198b6fdbbf087cd3d2dca0...  133.414531    163   
1  FLIR0011_jpg.rf.1e3a3f4c0e135fd1816eabc0a683a5...  133.547722    178   
2  FLIR0011_jpg.rf.20f0b633e8c37f08608654a804acd9...  133.536324    173   
3  FLIR0011_jpg.rf.255969173f316f8869269e225db745...  133.604127    167   
4  FLIR0011_jpg.rf.3e0bde0248a580b21a7e9d6e7f80c7...  133.407189    164   

       std_T       var_T  hotspot_ratio  gradient_mean  
0   9.470500   89.690367       0.035186       2.957933  
1  10.028307  100.566937       0.030477       3.704715  
2   9.888411   97.780669       0.028259       3.480412  
3   9.833405   96.695863       0.029082       3.533051  
4   9.488668   90.034825       0.034187       2.869356  
            mean_T        max_T        std_T        var_T  hotspot_ratio  \
count  4260.000000  4260.000000  4260.000000  4260.000000    4260.000000   
mean    112.286474   178.882864    15.767411   308.9943

In [21]:
q1 = df["mean_T"].quantile(0.25)
q2 = df["mean_T"].quantile(0.50)
q3 = df["mean_T"].quantile(0.75)

def assign_R(x):
    if x <= q1:
        return 1
    elif x <= q2:
        return 2
    elif x <= q3:
        return 3
    else:
        return 4

df["R_label"] = df["mean_T"].apply(assign_R)

In [22]:
df["Q_score"] = (
    df["hotspot_ratio"] * 0.6 +
    df["gradient_mean"] * 0.4
)

In [23]:
q1 = df["Q_score"].quantile(0.25)
q2 = df["Q_score"].quantile(0.50)
q3 = df["Q_score"].quantile(0.75)

def assign_Q(x):
    if x <= q1:
        return 1
    elif x <= q2:
        return 2
    elif x <= q3:
        return 3
    else:
        return 4

df["Q_label"] = df["Q_score"].apply(assign_Q)

In [24]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split

features = ["mean_T", "max_T", "std_T", "var_T", "hotspot_ratio", "gradient_mean"]

X = df[features]

# Train R model
y_R = df["R_label"]

X_train, X_test, y_train, y_test = train_test_split(X, y_R, test_size=0.2, random_state=42)

r_model = RandomForestClassifier(n_estimators=200)
r_model.fit(X_train, y_train)

print("R Accuracy:", r_model.score(X_test, y_test))

R Accuracy: 0.9988262910798122


In [26]:
import joblib

joblib.dump(r_model, "r_model.pkl")


['r_model.pkl']